In [8]:
import sagemaker, boto3, json
from sagemaker.model import Model

role = "arn:aws:iam::371087393859:role/defaultrole"
bucket = "ir-sagemaker"

session = boto3.Session(profile_name="lprofile", region_name="us-east-1")

sm_session = sagemaker.Session(boto_session=session, default_bucket=bucket)
region = sm_session.boto_region_name
container_version = '0.33.0-lmi15.0.0-cu128'

container_uri = f'763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{container_version}'
instance_type = "ml.g5.12xlarge"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/xdg-ubuntu/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/lbrenap/.config/sagemaker/config.yaml


/home/lbrenap/miniconda3/envs/legalpacaenv/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:

env = {
    "HF_MODEL_ID": "hltcoe/Rank-K-32B",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "TENSOR_PARALLEL_DEGREE": "max",
}

model = Model(
    image_uri=container_uri,
    role=role,
    env=env,
    sagemaker_session=sm_session,
)

endpoint_name = sagemaker.utils.name_from_base("rankk-vllm")
print(endpoint_name)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    endpoint_name=endpoint_name,
    container_startup_health_check_timeout = 1800,
)

print("InService as:", endpoint_name)


In [9]:
endpoint_name = "rankk-vllm-2025-09-03-17-29-50-270"
smr_client = boto3.client('sagemaker-runtime')

desc = sm_session.describe_endpoint(endpoint_name)
print("Status:", desc["EndpointStatus"])
print("ARN:", desc["EndpointArn"])

body = {
    "messages": [
        {"role": "user", "content": "Name places to visit in the US"}
    ],
    "temperature": 0.7,
    "max_tokens": 256,
    "stream": True,
}

resp = smr_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    Body=json.dumps(body),
    ContentType='application/json',
)

Status: InService
ARN: arn:aws:sagemaker:us-east-1:371087393859:endpoint/rankk-vllm-2025-09-03-17-29-50-270


In [3]:
print("Response:", end=' ', flush=True)
full_response = ""

for event in resp['Body']:
    if 'PayloadPart' in event:
        chunk = event['PayloadPart']['Bytes'].decode()

        try:
            if chunk.startswith('data: '):
                data = json.loads(chunk[6:])  # Skip "data: " prefix
            else:
                data = json.loads(chunk)

            if 'choices' in data and len(data['choices']) > 0:
                if 'delta' in data['choices'][0] and 'content' in data['choices'][0]['delta']:
                    token_text = data['choices'][0]['delta']['content']
                    full_response += token_text
                    print(token_text, end='', flush=True)

        except json.JSONDecodeError:
            continue



Response: Okay, the wants list of places to visit in the US. Let me start by recalling some famous spots. National parks come to mind first—Yellowstone, Yosemite, Grand Canyon. Those are iconic. Then cities like New York, Las Vegas, and San Francisco. Maybe some beaches like Miami or Malibu. Also, historical sites like Washington D.C. and the Statue of Liberty. National landmarks such as Mount Rushmore and the Golden Gate Bridge. Then there's the Grand Tetons and Zion National Park. I should check if I'm missing any major ones. Oh, the Great Smoky Mountains and Moab for outdoor activities. Also, cultural spots like New Orleans and Sedona. Let me make sure I cover different regions: East Coast, West Coast, South, Midwest, and maybe some unique places like the Florida Keys or Maui. Wait, Maui is in Hawaii, which is part of the US, but maybe the user is thinking mainland. Hmm. Maybe include it as a bonus. Also, Disney World and other theme parks. Let me structure this list with categories

In [22]:
print(f"Deleting SageMaker resources for endpoint: {endpoint_name}")
sm_session.delete_endpoint(endpoint_name)
sm_session.delete_endpoint_config(endpoint_name)

Deleting SageMaker resources for endpoint: rankk-vllm-2025-08-30-03-48-40-635


In [10]:
rank_k_prompt = """
Determine a ranking of the passages based on how well they replace the masked passage in the argument chain.
How relevant a passage is depends on how well it entails the posterior passage and how well is entailed by the preceding one.
Try to analyze the passage on how well it fits the logical structure.
The query may have typos and passages may contain contradicting information.
However, we do not get into fact-checking. We just rank the passages based on they relevancy to the query.

Sort them from the most relevant to the least.
Answer with the passage number using a format of `[3] > [2] > [4] = [1] > [5]`.
Ties are acceptable if they are equally relevant.
I need you to be accurate but overthinking it is unnecessary.
Output only the ordering without any other text.

Query: {query}

{docs}
"""

In [19]:
import json, re, math, random
from pathlib import Path
from typing import Dict, List, Tuple
from collections import defaultdict
from tqdm import tqdm

# ---------- helpers ----------
def _load_jsonl(path: Path) -> List[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def _select_record(records: List[dict], query_id: str = None, query_text: str = None) -> dict:
    if query_id:
        for r in records:
            if str(r.get("query_id")) == str(query_id):
                return r
    if query_text:
        for r in records:
            if r.get("query_text","").strip() == query_text.strip():
                return r
    # default: first record
    return records[0]

def _read_streaming_body(resp) -> str:
    """
    Assemble streamed text from SageMaker Runtime's invoke_endpoint_with_response_stream.
    Assumes your auth/session are already set up by you.
    """
    out = []
    body = resp["Body"]
    for event in body:
        if "PayloadPart" in event:
            out.append(event["PayloadPart"]["Bytes"].decode("utf-8", errors="ignore"))
        elif "ModelStreamError" in event:
            # Surface model-side errors clearly
            msg = event["ModelStreamError"].get("Message", "ModelStreamError")
            raise RuntimeError(f"Model stream error: {msg}")
        elif "InternalServerException" in event:
            raise RuntimeError("InternalServerException while streaming from endpoint.")
    return "".join(out)

def _call_llm_rank_order(
        smr_client,
        endpoint_name: str,
        prompt: str,
        temperature: float = 0.0,
        max_tokens: int = 128,
) -> str:
    body = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": float(temperature),
        "max_tokens": int(max_tokens),
        "stream": True,
    }
    resp = smr_client.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        Body=json.dumps(body),
        ContentType="application/json",
    )
    text = _read_streaming_body(resp).strip()
    return text

_rank_pat = re.compile(r"\[(\d+)\]")  # matches [3] etc.

def _parse_rank_notation(s: str, n_local: int) -> List[List[int]]:
    """
    Parse outputs like:  [3] > [2] > [4] = [1] > [5]
    Return groups of local indices (1..n_local), where inner lists are ties.
    Robust to minor deviations; falls back to a simple sequence if needed.
    """
    # keep only brackets, digits, '=', '>' to be tolerant
    cleaned = re.sub(r"[^\[\]\d=>]", "", s)
    groups: List[List[int]] = []
    for seg in cleaned.split(">"):
        ids = [int(x) for x in _rank_pat.findall(seg)]
        ids = [i for i in ids if 1 <= i <= n_local]
        if ids:
            groups.append(ids)
    if not groups:
        # last resort: just all numbers in order they appear
        ids = [int(x) for x in re.findall(r"\d+", s)]
        ids = [i for i in ids if 1 <= i <= n_local]
        groups = [[i] for i in ids] if ids else []
    return groups

def _borda_from_groups(groups: List[List[int]], n_local: int) -> Dict[int, float]:
    """
    Convert listwise ordering (with ties) into Borda scores.
    For N items, rank r gets N - r points. Ties share the average of their positions.
    Returns dict over local index -> score.
    """
    scores = defaultdict(float)
    pos = 1
    for g in groups:
        m = len(g)
        # average points over ranks pos..pos+m-1
        avg = sum(n_local - r for r in range(pos, pos + m)) / float(m)
        for item in g:
            scores[item] += avg
        pos += m
    # ensure every item included
    for i in range(1, n_local + 1):
        scores.setdefault(i, 0.0)
    return scores

def _build_batches(n: int, k: int = 8, stride: int = 6) -> List[List[int]]:
    """
    Sliding windows over 0..n-1 of size k with overlap (stride).
    Ensures last window includes the tail.
    Returns list of lists of GLOBAL indices.
    """
    batches = []
    if n <= k:
        return [list(range(n))]
    for start in range(0, n, stride):
        end = min(n, start + k)
        if end - start < k:
            start = max(0, n - k)
            end = n
        batches.append(list(range(start, end)))
        if end == n:
            break
    return batches


def _fill_rankk_template(template: str, query_text: str, block: str) -> str:
    """Fill your rank_k_prompt robustly: prefer {passages}, fall back to {docs}."""
    if "{passages}" in template:
        return template.format(query=query_text, passages=block)
    if "{docs}" in template:
        return template.format(query=query_text, docs=block)
    # last resort: just append passages and fill {query}
    return (template.rstrip() + "\n\n" + block).format(query=query_text)

def _iter_jsonl(path: Path):
    """Stream JSONL, skipping empty lines; raises with line no. if a line is invalid."""
    with path.open("r", encoding="utf-8") as f:
        for i, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise RuntimeError(f"{path} invalid JSON on line {i}: {line[:120]!r}") from e

# ---------- main reranker ----------
def rerank_topk50_rankk_all(
        smr_client,
        endpoint_name: str,
        rank_k_prompt: str,
        topk_path: Path = Path("retrieval_results/retrieved/topk_50.jsonl"),
        out_dir: Path = Path("retrieval_results/reranked"),
        window_k: int = 8,
        stride: int = 6,
        snippet_limit: int = 480,
        temperature: float = 0.0,
        max_tokens: int = 128,
        run_name: str = "rankk_top50",
):
    """
    Rerank ALL queries found in topk_50.jsonl with Rank-K style listwise batching.
    Writes:
      - rankk_top50.jsonl    (one line per query with final order)
      - rankk_top50.trec     (TREC run for all qids)
      - rankk_top50_judgments.jsonl  (one line per query with batch-level LLM outputs)
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    out_jsonl = out_dir / "rankk_top50.jsonl"
    out_trec  = out_dir / "rankk_top50.trec"
    out_judg  = out_dir / "rankk_top50_judgments.jsonl"

    # open once for the whole run
    fj = out_jsonl.open("w", encoding="utf-8")
    ft = out_trec.open("w", encoding="utf-8")
    fg = out_judg.open("w", encoding="utf-8")

    try:
        for i, rec in tqdm(enumerate(_iter_jsonl(topk_path))):
            if i > 5:
                break
            qid   = str(rec.get("query_id", "q0"))
            qtext = rec.get("query_text", "")
            items = rec.get("results", []) or []
            if not items:
                continue

            # compact text for prompts
            for it in items:
                t = str(it.get("text", ""))
                it["_prompt_text"] = t[:snippet_limit].replace("\n", " ")

            n = len(items)
            batches = _build_batches(n, k=window_k, stride=stride)

            agg_scores = defaultdict(float)
            judgments = []

            for bidx, batch in enumerate(batches):
                lines, local_to_global = [], {}
                for local_i, gi in enumerate(batch, start=1):
                    local_to_global[local_i] = gi
                    lines.append(f"[{local_i}] {items[gi]['_prompt_text']}")

                block  = "\n".join(lines)
                prompt = _fill_rankk_template(rank_k_prompt, qtext, block)

                llm_out = _call_llm_rank_order(
                    smr_client=smr_client,
                    endpoint_name=endpoint_name,
                    prompt=prompt,
                    temperature=temperature,
                    max_tokens=max_tokens,
                )

                groups = _parse_rank_notation(llm_out, n_local=len(batch))
                borda  = _borda_from_groups(groups, n_local=len(batch))

                for local_i, score in borda.items():
                    gi = local_to_global[local_i]
                    agg_scores[gi] += float(score)

                judgments.append({
                    "query_id": qid,
                    "batch_index": bidx,
                    "global_indices": batch,
                    "llm_output": llm_out,
                    "parsed_groups": groups,
                    "local_borda": {int(k): float(v) for k, v in borda.items()},
                })

            # final sort for this query
            order = sorted(
                range(n),
                key=lambda gi: (-agg_scores.get(gi, 0.0),
                                int(items[gi].get("rank", 10**9)),
                                -float(items[gi].get("score", 0.0))),
            )

            # write JSONL (one line per query)
            fj.write(json.dumps({
                "method": "rankk_listwise_borda",
                "run_name": run_name,
                "query_id": qid,
                "query_text": qtext,
                "k": 50,
                "window_k": window_k,
                "stride": stride,
                "final": [
                    {
                        "rerank_rank": r,
                        "agg_score": float(agg_scores[gi]),
                        "key": items[gi]["key"],
                        "docid": items[gi]["docid"],
                        "text": items[gi].get("text", ""),
                        "orig_rank": int(items[gi].get("rank", r)),
                        "orig_score": float(items[gi].get("score", 0.0)),
                    }
                    for r, gi in enumerate(order, start=1)
                ],
            }, ensure_ascii=False) + "\n")

            # write TREC lines for this query
            for r, gi in enumerate(order, start=1):
                ft.write(f"{qid} Q0 {items[gi]['docid']} {r} {float(agg_scores[gi]):.6f} {run_name}\n")

            # write judgments trace (one line per query)
            fg.write(json.dumps({
                "query_id": qid,
                "query_text": qtext,
                "window_k": window_k,
                "stride": stride,
                "batches": judgments,
            }, ensure_ascii=False) + "\n")

        print(f"[saved] {out_jsonl}")
        print(f"[saved] {out_trec}")
        print(f"[saved] {out_judg}")
    finally:
        fj.close(); ft.close(); fg.close()



In [20]:
# processes every query present in topk_50.jsonl
rerank_topk50_rankk_all(
    smr_client=smr_client,
    endpoint_name=endpoint_name,
    rank_k_prompt=rank_k_prompt,   # your template with {passages}
    topk_path=Path("retrieval_results/retrieved/topk_50.jsonl"),
    out_dir=Path("retrieval_results/reranked"),
    window_k=8, stride=6, snippet_limit=480,
    temperature=0.0, max_tokens=128,
    run_name="rankk_top50",
)


# The final files will be in retrieval_results/reranked/


[saved] retrieval_results/reranked/rankk_top50.jsonl
[saved] retrieval_results/reranked/rankk_top50.trec
[saved] retrieval_results/reranked/rankk_top50_judgments.jsonl


In [42]:
from __future__ import annotations
import json
from pathlib import Path
from typing import Dict, Iterable, List, Optional
from statistics import mean

def _safe_mean(xs: List[float]) -> float:
    return mean(xs) if xs else float("nan")

def evaluate_reranked_jsonl(
        path: Path,
        ks: Iterable[int] = (1, 5, 10, 20, 50),
        verbose_examples: int = 5,
) -> Dict:
    """
    Args:
        path: JSONL file; each line has {query_id, query_text, query_docid, k, results:[{rank,is_positive,...}, ...]}
        ks:   Cutoffs to report Hit@K for. Include 50 for Hit@50 (default does).
    Returns:
        metrics dict with hit_rate@K, mrr, mean_rank_found, queries_with_positive_in_topk, per_doc aggregates.
    """
    ks = sorted(set(int(k) for k in ks))
    total = 0
    hits = {k: 0 for k in ks}
    rr_sum = 0.0
    ranks_found: List[int] = []
    per_doc = {}  # docid -> {num_queries, ranks[], rrs[]}
    examples: List[Dict] = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            total += 1

            qid = rec.get("query_id")
            qtxt = rec.get("query_text", "")
            qdoc = rec.get("query_docid", None)
            results = rec.get("results", []) or []

            # find best (lowest) rank among positives in the top-k list
            pos_ranks = [int(it.get("rank", 10**9)) for it in results if it.get("is_positive")]
            pos_rank: Optional[int] = min(pos_ranks) if pos_ranks else None

            # metrics per query
            for k in ks:
                if pos_rank is not None and pos_rank <= k:
                    hits[k] += 1
            if pos_rank is not None:
                rr_sum += 1.0 / pos_rank
                ranks_found.append(pos_rank)
            else:
                rr_sum += 0.0  # explicit: MRR contribution is 0 when not found

            # per-doc aggregates
            if qdoc not in per_doc:
                per_doc[qdoc] = {"num_queries": 0, "ranks": [], "rrs": []}
            per_doc[qdoc]["num_queries"] += 1
            if pos_rank is not None:
                per_doc[qdoc]["ranks"].append(float(pos_rank))
                per_doc[qdoc]["rrs"].append(1.0 / pos_rank)

            # collect a few examples for quick sanity checks
            if len(examples) < verbose_examples:
                examples.append({
                    "query_id": qid,
                    "query_docid": qdoc,
                    "pos_rank": pos_rank,
                    "found_at": {k: (pos_rank is not None and pos_rank <= k) for k in ks},
                    "top_keys": [it.get("key") for it in results[:3]],
                })

    # macro
    hit_rate = {k: (hits[k] / total if total else float("nan")) for k in ks}
    mrr = (rr_sum / total) if total else float("nan")
    mean_rank_found = _safe_mean(ranks_found)
    metrics = {
        "num_queries": total,
        "hit_rate": hit_rate,
        "mrr": mrr,
        "mean_rank_found": mean_rank_found,
        "queries_with_positive_in_topk": len(ranks_found),
    }

    # per-doc aggregates
    per_doc_stats = {}
    for did, b in per_doc.items():
        per_doc_stats[did] = {
            "num_queries": b["num_queries"],
            "avg_rank": _safe_mean(b["ranks"]),
            "mrr": _safe_mean(b["rrs"]),
        }
    metrics["per_doc"] = per_doc_stats

    # quick console report
    print("\n=== Macro metrics (from reranked top-k) ===")
    for k in ks:
        print(f"Hit@{k:>2} = {metrics['hit_rate'][k]:.4f}" if metrics['num_queries'] else f"Hit@{k:>2} = NaN")
    print(f"MRR     = {metrics['mrr']:.4f}" if metrics['num_queries'] else "MRR     = NaN")
    print(f"Mean rank (found) = {metrics['mean_rank_found']:.2f}" if ranks_found else "Mean rank (found) = NaN")
    print(f"Queries with positive in top-k = {metrics['queries_with_positive_in_topk']} of {metrics['num_queries']}")

    print("\n=== Examples ===")
    for ex in examples:
        print(f"- {ex['query_id']} | doc={ex['query_docid']} | pos_rank={ex['pos_rank']} | found={ex['found_at']}")
        print(f"  top3: {ex['top_keys']}")

    return metrics

if __name__ == "__main__":
    # Point this to your reranked file (same format as your sample, but with k=50).
    jsonl_path = Path("retrieval_results/retrieved/topk_50.jsonl")
    evaluate_reranked_jsonl(jsonl_path, ks=(1, 5, 10, 20, 50))



=== Macro metrics (from reranked top-k) ===
Hit@ 1 = 0.0807
Hit@ 5 = 0.2153
Hit@10 = 0.3147
Hit@20 = 0.4555
Hit@50 = 0.6729
MRR     = 0.1560
Mean rank (found) = 15.75
Queries with positive in top-k = 325 of 483

=== Examples ===
- R2011_France Télécom SA v European Commission:0 | doc=R2011_France Télécom SA v European Commission | pos_rank=None | found={1: False, 5: False, 10: False, 20: False, 50: False}
  top3: ['R2011_France Télécom SA v European Commission:93', 'R2011_France Télécom SA v European Commission:140', 'R2011_France Télécom SA v European Commission:145']
- R2011_France Télécom SA v European Commission:1 | doc=R2011_France Télécom SA v European Commission | pos_rank=None | found={1: False, 5: False, 10: False, 20: False, 50: False}
  top3: ['R2011_France Télécom SA v European Commission:82', 'R2011_France Télécom SA v European Commission:76', 'R2011_France Télécom SA v European Commission:175']
- R2011_France Télécom SA v European Commission:2 | doc=R2011_France Télécom 

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:25                                                                                   │
│                                                                                                  │
│   22 reranked_keys = {}                                                                          │
│   23 with RERANK_JSONL.open("r", encoding="utf-8") as f:                                         │
│   24 │   for line in f:                                                                          │
│ ❱ 25 │   │   rec = json.loads(line)                                                              │
│   26 │   │   qid = str(rec["query_id"])                                                          │
│   27 │   │   finals = sorted(rec["final"], key=lambda x: int(x["rerank_rank"]))                  │
│   28 │   │   reranked_keys[qid] = [it["key"] for it in finals]  # <-- use "key", not "docid"     │
│                                                                                                  │
│ /home/lbrenap/miniconda3/envs/legalpacaenv/lib/python3.11/json/__init__.py:346 in loads          │
│                                                                                                  │
│   343 │   if (cls is None and object_hook is None and                                            │
│   344 │   │   │   parse_int is None and parse_float is None and                                  │
│   345 │   │   │   parse_constant is None and object_pairs_hook is None and not kw):              │
│ ❱ 346 │   │   return _default_decoder.decode(s)                                                  │
│   347 │   if cls is None:                                                                        │
│   348 │   │   cls = JSONDecoder                                                                  │
│   349 │   if object_hook is not None:                                                            │
│                                                                                                  │
│ /home/lbrenap/miniconda3/envs/legalpacaenv/lib/python3.11/json/decoder.py:337 in decode          │
│                                                                                                  │
│   334 │   │   containing a JSON document).                                                       │
│   335 │   │                                                                                      │
│   336 │   │   """                                                                                │
│ ❱ 337 │   │   obj, end = self.raw_decode(s, idx=_w(s, 0).end())                                  │
│   338 │   │   end = _w(s, end).end()                                                             │
│   339 │   │   if end != len(s):                                                                  │
│   340 │   │   │   raise JSONDecodeError("Extra data", s, end)                                    │
│                                                                                                  │
│ /home/lbrenap/miniconda3/envs/legalpacaenv/lib/python3.11/json/decoder.py:353 in raw_decode      │
│                                                                                                  │
│   350 │   │                                                                                      │
│   351 │   │   """                                                                                │
│   352 │   │   try:                                                                               │
│ ❱ 353 │   │   │   obj, end = self.scan_once(s, idx)                                              │
│   354 │   │   except StopIteration as err:                                                       │
│   355 │   │   │   raise JSONDecodeError("Expecting value", s, err.value) from None               │
│   356 │   │   return obj, end                              